# 02. clean

## 0. setup

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

## 1. config

In [2]:
# cartel
in_cartel    = data_raw / 'cartel' / 'manual' / 'cartel_manual_v8.xlsx'
in_nace_isic = data_raw / 'external' / 'nace2_isic4.txt'

# patents
in_pat      = data_interim / 'pat_data.parquet'
in_ipc_isic = data_raw / 'external' / 'ipc4_to_isic_rev4_3.txt'

# panel country set: fixed EEA + GB
eea = set(pd.read_excel(in_cartel, sheet_name='ref_ctry_timeline')['ctry_iso'])

# outputs
out_treat = data_interim / 'treat_cartel.parquet'
out_clong = data_interim / 'cartel_long.parquet'
out_pat   = data_interim / 'pat_isic3.parquet'

print(f'cartel : {in_cartel.name} + {in_nace_isic.name} -> {out_treat.name}, {out_clong.name}')
print(f'patents: {in_pat.name} + {in_ipc_isic.name} -> {out_pat.name}')
print(f'countries: {len(eea)}')

cartel : cartel_manual_v8.xlsx + nace2_isic4.txt -> treat_cartel.parquet, cartel_long.parquet
patents: pat_data.parquet + ipc4_to_isic_rev4_3.txt -> pat_isic3.parquet
countries: 31


## 2. cartel

### 2.1 infringements

In [3]:
cart = pd.read_excel(in_cartel, sheet_name='cartels')
n0 = len(cart)

# analysis set: blank exclude_reason
cart = cart[cart['exclude_reason'].isna()]

# industry and decision year must be the same across the segments of an infringement
for c in ['nace_code', 'decision_year']:
    assert (cart.groupby('infringement_id')[c].nunique() == 1).all(), f'{c} varies within infringement'

# collapse segments: one row per infringement
infr = cart.groupby('infringement_id', as_index=False)[['nace_code', 'decision_year']].first()

print(f'cartels: {n0} rows -> {len(cart)} not excluded -> {len(infr)} infringements')

cartels: 215 rows -> 163 not excluded -> 158 infringements


### 2.2 participants

In [4]:
firms = pd.read_excel(in_cartel, sheet_name='firms')
firms.columns = firms.columns.str.strip()
n0 = len(firms)

# keep participants of kept infringements (drops firms of excluded cases) and drop firm_exclude rows
part = firms[firms['infringement_id'].isin(infr['infringement_id']) & firms['firm_exclude'].isna()]
n1 = len(part)

# dates: present and start <= end
assert (part['start_date'] <= part['end_date']).all(), 'missing dates or start > end'

# keep EEA countries
part = part.loc[part['ctry_iso'].isin(eea), ['infringement_id', 'ctry_iso', 'start_date', 'end_date']]

print(f'participants: {n0} rows -> {n1} kept -> {len(part)} in EEA '
      f'| infringements with an EEA participant: {part["infringement_id"].nunique()} / {len(infr)}')

participants: 1818 rows -> 1605 kept -> 1212 in EEA | infringements with an EEA participant: 150 / 158


### 2.3 nace to isic crosswalk

In [5]:
nace_isic = pd.read_csv(in_nace_isic, dtype=str)
nace_isic['nace']  = nace_isic['NACE2code'].str.replace('.', '', regex=False)
nace_isic['isic3'] = nace_isic['ISIC4code'].str[:3]

# split multi-code cells, one row per code
codes = infr[['infringement_id', 'nace_code']].assign(nace=infr['nace_code'].str.split(',')).explode('nace')
codes['nace'] = codes['nace'].str.strip().str[1:]                     # drop section letter: C2932 -> 2932

# 3/4-digit codes: table lookup, never truncation
fine = codes[codes['nace'].str.len() >= 3].merge(nace_isic[['nace', 'isic3']], on='nace', how='left').assign(treat_2d=0)
assert fine['isic3'].notna().all(), f'codes not in crosswalk: {fine.loc[fine["isic3"].isna(), "nace"].tolist()}'

# 2-digit codes: every ISIC3 group in the division, flagged treat_2d = 1
groups = nace_isic.loc[nace_isic['nace'].str.len() == 3, 'isic3'].drop_duplicates()
coarse = (codes[codes['nace'].str.len() == 2]
            .merge(pd.DataFrame({'isic3': groups, 'nace': groups.str[:2]}), on='nace').assign(treat_2d=1))

# infringement x isic3; treat_2d = 1 only if the pair is reached via a 2-digit code alone
imap = pd.concat([fine, coarse]).groupby(['infringement_id', 'isic3'], as_index=False)['treat_2d'].min()

print(f'codes: {len(codes)} ({(codes["nace"].str.len() == 2).sum()} at 2 digits) '
      f'-> {len(imap)} infringement x isic3 pairs | {imap["isic3"].nunique()} isic3 groups')

codes: 186 (15 at 2 digits) -> 211 infringement x isic3 pairs | 59 isic3 groups


### 2.4 annual exposure

In [6]:
# expand each participation to days (start and end included), map to isic3
days = part.assign(day=[pd.date_range(s, e) for s, e in zip(part['start_date'], part['end_date'])]).explode('day')
days = days.merge(imap, on='infringement_id')
days['year'] = days['day'].dt.year

k = ['isic3', 'ctry_iso', 'year']

# cartel_long: infringement x isic3 x country x year (decision cohorts built in 03)
clong = (days[['infringement_id'] + k + ['treat_2d']].drop_duplicates()
           .merge(infr[['infringement_id', 'decision_year']], on='infringement_id'))

# cell-year exposure: distinct cartelised days / days in year (overlapping participants counted once)
treat = days.drop_duplicates(['isic3', 'ctry_iso', 'day']).groupby(k).size().rename('n_days').reset_index()
treat['expo']    = treat['n_days'] / (365 + (treat['year'] % 4 == 0))   # leap years (1969-2022: every 4th)
treat['treated'] = (treat['expo'] >= 0.5).astype(int)                    # main rule; expo > 0 as robustness in 03

# treat_2d = 1 only if every infringement active in the cell-year reaches it via a 2-digit code
treat = treat.merge(clong.groupby(k, as_index=False)['treat_2d'].min(), on=k).drop(columns='n_days')

print(f'cartel_long: {len(clong):,} rows | treat_cartel: {len(treat):,} cell-years with expo > 0, '
      f'{treat["treated"].sum():,} treated | {treat.groupby(["isic3", "ctry_iso"]).ngroups} cells')
print(f'treated via 2-digit only: {treat.loc[treat["treated"] == 1, "treat_2d"].mean():.1%} '
      f'| years {treat["year"].min()}-{treat["year"].max()}')
del days; gc.collect()

cartel_long: 5,500 rows | treat_cartel: 3,627 cell-years with expo > 0, 3,229 treated | 346 cells
treated via 2-digit only: 27.2% | years 1969-2022


28

### 2.5 save

In [7]:
clong.to_parquet(out_clong, index=False)
treat.to_parquet(out_treat, index=False)
print(f'saved: {out_clong.name} ({len(clong):,}) | {out_treat.name} ({len(treat):,})')

saved: cartel_long.parquet (5,500) | treat_cartel.parquet (3,627)


## 3. patents

### 3.1 load

In [8]:
pat = pd.read_parquet(in_pat, columns=['appln_id', 'applt_id', 'ctry_code', 'app_share', 'prio_year', 'ipc', 'cit_fwd_3yr'])
print(f'pat_data: {len(pat):,} rows | {pat["appln_id"].nunique():,} applications | {pat["prio_year"].min()}-{pat["prio_year"].max()}')

pat_data: 21,341,551 rows | 4,682,001 applications | 1961-2025


### 3.2 applicant weights

In [9]:
# one row per application x applicant
app = pat[['appln_id', 'applt_id', 'ctry_code', 'app_share']].drop_duplicates(['appln_id', 'applt_id'])
n_all = app['appln_id'].nunique()

# keep EEA applicants with their app_share
app = app[app['ctry_code'].isin(eea)]

print(f'EEA applicants: {app["appln_id"].nunique():,} / {n_all:,} applications | EEA share of mass {app["app_share"].sum() / n_all:.1%}')

EEA applicants: 1,980,311 / 4,682,001 applications | EEA share of mass 41.8%


### 3.3 ipc4 weights

In [10]:
alp = pd.read_csv(in_ipc_isic, dtype={'ipc4': str, 'isic_rev4_3': str}).rename(columns={'isic_rev4_3': 'isic3', 'probability_weight': 'w_alp'})

# post-2018 IPC subclasses (not in ALP) -> predecessor subclass
ipc_new = {'G06V': 'G06K', 'G16H': 'G06F', 'G16B': 'G06F', 'G16Z': 'G06F',
           'H10K': 'H01L', 'H10N': 'H01L', 'H10B': 'H01L', 'F24S': 'F24J'}
assert set(ipc_new.values()) <= set(alp['ipc4'])

# IPC4 = first 4 characters; one row per application x distinct IPC4 (EEA-applicant applications only)
ipc = pat.loc[pat['appln_id'].isin(app['appln_id']), ['appln_id', 'prio_year', 'ipc', 'cit_fwd_3yr']].copy()
ipc['ipc4'] = ipc['ipc'].str.replace(' ', '', regex=False).str[:4].replace(ipc_new)
ipc = ipc.drop(columns='ipc').drop_duplicates(['appln_id', 'ipc4'])

# each distinct IPC4 of an application gets 1/n
ipc['w_ipc'] = 1 / ipc.groupby('appln_id')['ipc4'].transform('size')

# IPC4 not in ALP are dropped
matched = ipc['ipc4'].isin(alp['ipc4'])
print(f'IPC4 mass matched to ALP: {ipc.loc[matched, "w_ipc"].sum() / ipc["appln_id"].nunique():.2%} '
      f'| unmatched: {ipc.loc[~matched, "ipc4"].value_counts().head(5).to_dict()}')
ipc = ipc[matched]

del pat; gc.collect()

IPC4 mass matched to ALP: 99.97% | unmatched: {'G16C': 341, 'B64U': 265, 'H10D': 187, 'F24V': 147, 'F24T': 147}


43

### 3.4 assign to industry

In [11]:
# weight = app_share x 1/n IPC4
rows = app[['appln_id', 'ctry_code', 'app_share']].merge(ipc, on='appln_id')
rows['w']      = rows['app_share'] * rows['w_ipc']
rows['w_cit3'] = rows['w'] * rows['cit_fwd_3yr']

# collapse to country x year x IPC4, then spread over ISIC3 with ALP probabilities
cell = rows.groupby(['ctry_code', 'prio_year', 'ipc4'], as_index=False)[['w', 'w_cit3']].sum().merge(alp, on='ipc4')
cell['pat_frac']  = cell['w'] * cell['w_alp']
cell['cit3_frac'] = cell['w_cit3'] * cell['w_alp']

# outcomes by isic3 x country x year: pat_frac, cit3_frac
panel = (cell.groupby(['isic3', 'ctry_code', 'prio_year'], as_index=False)[['pat_frac', 'cit3_frac']].sum()
             .rename(columns={'ctry_code': 'ctry_iso', 'prio_year': 'year'}))
assert np.isclose(panel['pat_frac'].sum(), rows['w'].sum()), 'mass lost'

print(f'panel: {len(panel):,} cell-years (non-zero only) | {panel["isic3"].nunique()} isic3 | '
      f'{panel["ctry_iso"].nunique()} countries | {panel["year"].min()}-{panel["year"].max()}')
del rows, cell; gc.collect()

panel: 157,457 cell-years (non-zero only) | 212 isic3 | 31 countries | 1961-2025


0

### 3.5 diagnostics

In [12]:
# series by priority year: basis for the sample cutoffs set in 03
by_year = panel.groupby('year')[['pat_frac', 'cit3_frac']].sum()
by_year['cit3_per_pat'] = by_year['cit3_frac'] / by_year['pat_frac']
with pd.option_context('display.max_rows', None):
    print(by_year.round(3).to_string())

       pat_frac  cit3_frac  cit3_per_pat
year                                    
1961      1.000      0.000         0.000
1968      1.000      0.000         0.000
1969      1.000      0.000         0.000
1975      1.000      0.000         0.000
1977   2037.583    577.750         0.284
1978   6666.667   1979.917         0.297
1979  10640.098   3354.774         0.315
1980  13212.907   4163.345         0.315
1981  14846.032   5085.607         0.343
1982  15665.355   5409.571         0.345
1983  17856.357   6178.524         0.346
1984  19484.749   7466.524         0.383
1985  21338.976   8353.238         0.391
1986  22429.071   9489.464         0.423
1987  25235.217  11139.636         0.441
1988  27316.543  11799.707         0.432
1989  28091.681  12635.124         0.450
1990  26611.068  12474.800         0.469
1991  26520.579  12709.495         0.479
1992  26523.933  13106.117         0.494
1993  27584.248  13389.050         0.485
1994  28953.846  15073.817         0.521
1995  30499.725 

### 3.6 save

In [13]:
panel.to_parquet(out_pat, index=False)
print(f'saved: {out_pat.name} ({len(panel):,} rows)')

del panel, app, ipc; gc.collect()

saved: pat_isic3.parquet (157,457 rows)


0